In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

In [2]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUT_DIR = BASE_DIR / "outputs" / "part2"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(PROCESSED_DIR / "audusd_part1_corrected.csv", parse_dates=["Date"])

print(df.shape)
df.head()

(179, 12)


,Date,spot,fx_return,rate_au,rate_us,infl_au,infl_us,actual_fx_change,uip_implied_change,ppp_implied_change,uip_deviation,ppp_deviation
0,2009-02-28,0.642219,0.005827,3.35,0.16,0.000363,0.004961,0.005827,-0.002658,0.004597,-0.008485,-0.001230
1,2009-03-31,0.691802,0.074370,3.25,0.17,0.000363,0.002429,0.074370,-0.002567,0.002065,-0.076937,-0.072304
2,2009-04-30,0.726691,0.049202,3.06,0.04,0.001659,0.002493,0.049202,-0.002517,0.000834,-0.051719,-0.048368
3,2009-05-31,0.801025,0.097391,3.00,0.14,0.001657,0.002885,0.097391,-0.002383,0.001228,-0.099774,-0.096163
4,2009-06-30,0.806192,0.006429,3.00,0.17,0.001654,0.008553,0.006429,-0.002358,0.006899,-0.008787,0.000471


In [4]:
part2 = df[
    (df["Date"] >= "2009-01-01") &
    (df["Date"] <= "2019-12-31")
].dropna(
    subset=[
        "actual_fx_change",
        "uip_implied_change",
        "ppp_implied_change"
    ]
).copy()

print(part2.shape)
print(part2["Date"].min())
print(part2["Date"].max())

part2.head()

(131, 12)
2009-02-28 00:00:00
2019-12-31 00:00:00


,Date,spot,fx_return,rate_au,rate_us,infl_au,infl_us,actual_fx_change,uip_implied_change,ppp_implied_change,uip_deviation,ppp_deviation
0,2009-02-28,0.642219,0.005827,3.35,0.16,0.000363,0.004961,0.005827,-0.002658,0.004597,-0.008485,-0.001230
1,2009-03-31,0.691802,0.074370,3.25,0.17,0.000363,0.002429,0.074370,-0.002567,0.002065,-0.076937,-0.072304
2,2009-04-30,0.726691,0.049202,3.06,0.04,0.001659,0.002493,0.049202,-0.002517,0.000834,-0.051719,-0.048368
3,2009-05-31,0.801025,0.097391,3.00,0.14,0.001657,0.002885,0.097391,-0.002383,0.001228,-0.099774,-0.096163
4,2009-06-30,0.806192,0.006429,3.00,0.17,0.001654,0.008553,0.006429,-0.002358,0.006899,-0.008787,0.000471


In [5]:
part2[
    [
        "Date",
        "actual_fx_change",
        "uip_implied_change",
        "ppp_implied_change"
    ]
].head(12)

,Date,actual_fx_change,uip_implied_change,ppp_implied_change
0,2009-02-28,0.005827,-0.002658,0.004597
1,2009-03-31,0.074370,-0.002567,0.002065
2,2009-04-30,0.049202,-0.002517,0.000834
3,2009-05-31,0.097391,-0.002383,0.001228
4,2009-06-30,0.006429,-0.002358,0.006899
5,2009-07-31,0.035866,-0.002383,-0.004732
6,2009-08-31,0.008644,-0.002408,-0.000895
7,2009-09-30,0.047379,-0.002450,-0.002500
8,2009-10-31,0.019274,-0.002667,-0.000826
9,2009-11-30,0.018276,-0.002833,-0.001078


In [6]:
X_uip = sm.add_constant(part2["uip_implied_change"])
y = part2["actual_fx_change"]

uip_model = sm.OLS(y, X_uip).fit()

print(uip_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.066
Date:                Tue, 12 May 2026   Prob (F-statistic):              0.304
Time:                        20:13:03   Log-Likelihood:                 257.73
No. Observations:                 131   AIC:                            -511.5
Df Residuals:                     129   BIC:                            -505.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.0031      0

In [7]:
uip_results = pd.DataFrame({
    "term": uip_model.params.index,
    "coef": uip_model.params.values,
    "std_error": uip_model.bse.values,
    "p_value": uip_model.pvalues.values,
    "ci_low": uip_model.conf_int()[0].values,
    "ci_high": uip_model.conf_int()[1].values
})

uip_results

,term,coef,std_error,p_value,ci_low,ci_high
0,const,-0.003082,0.004724,0.515333,-0.012429,0.006265
1,uip_implied_change,-2.119239,2.052887,0.303853,-6.180925,1.942447


In [8]:
X_ppp = sm.add_constant(part2["ppp_implied_change"])
y = part2["actual_fx_change"]

ppp_model = sm.OLS(y, X_ppp).fit()

print(ppp_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.4710
Date:                Tue, 12 May 2026   Prob (F-statistic):              0.494
Time:                        20:13:21   Log-Likelihood:                 257.43
No. Observations:                 131   AIC:                            -510.9
Df Residuals:                     129   BIC:                            -505.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0009      0

In [9]:
ppp_results = pd.DataFrame({
    "term": ppp_model.params.index,
    "coef": ppp_model.params.values,
    "std_error": ppp_model.bse.values,
    "p_value": ppp_model.pvalues.values,
    "ci_low": ppp_model.conf_int()[0].values,
    "ci_high": ppp_model.conf_int()[1].values
})

ppp_results

,term,coef,std_error,p_value,ci_low,ci_high
0,const,0.000876,0.002996,0.770470,-0.005052,0.006804
1,ppp_implied_change,0.693403,1.010372,0.493765,-1.305641,2.692448


In [10]:
summary = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "alpha": [
        uip_model.params["const"],
        ppp_model.params["const"]
    ],
    "alpha_p_value": [
        uip_model.pvalues["const"],
        ppp_model.pvalues["const"]
    ],
    "beta": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "beta_p_value": [
        uip_model.pvalues["uip_implied_change"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "r_squared": [
        uip_model.rsquared,
        ppp_model.rsquared
    ],
    "n_obs": [
        int(uip_model.nobs),
        int(ppp_model.nobs)
    ]
})

summary

,model,alpha,alpha_p_value,beta,beta_p_value,r_squared,n_obs
0,UIP,-0.003082,0.515333,-2.119239,0.303853,0.008193,131
1,PPP,0.000876,0.770470,0.693403,0.493765,0.003638,131


In [11]:
theory_table = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "regression_equation": [
        "Δs_t = α + β(i_US,t - i_AUS,t) + ε_t",
        "Δs_t = α + β(π_US,t - π_AUS,t) + ε_t"
    ],
    "theoretical_alpha": [0, 0],
    "theoretical_beta": [1, 1],
    "null_hypothesis": [
        "H0: alpha = 0 and beta = 1",
        "H0: alpha = 0 and beta = 1"
    ],
    "alternative_hypothesis": [
        "H1: alpha != 0 and/or beta != 1",
        "H1: alpha != 0 and/or beta != 1"
    ]
})

theory_table

,model,regression_equation,theoretical_alpha,theoretical_beta,null_hypothesis,alternative_hypothesis
0,UIP,"Δs_t = α + β(i_US,t - i_AUS,t) + ε_t",0,1,H0: alpha = 0 and beta = 1,H1: alpha != 0 and/or beta != 1
1,PPP,"Δs_t = α + β(π_US,t - π_AUS,t) + ε_t",0,1,H0: alpha = 0 and beta = 1,H1: alpha != 0 and/or beta != 1


In [12]:
uip_beta_test = uip_model.t_test("uip_implied_change = 1")
ppp_beta_test = ppp_model.t_test("ppp_implied_change = 1")

print("UIP test beta = 1")
print(uip_beta_test)

print("\nPPP test beta = 1")
print(ppp_beta_test)

UIP test beta = 1
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -2.1192      2.053     -1.519      0.131      -6.181       1.942

PPP test beta = 1
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.6934      1.010     -0.303      0.762      -1.306       2.692


In [13]:
uip_joint_test = uip_model.f_test("const = 0, uip_implied_change = 1")
ppp_joint_test = ppp_model.f_test("const = 0, ppp_implied_change = 1")

print("UIP joint test: alpha = 0 and beta = 1")
print(uip_joint_test)

print("\nPPP joint test: alpha = 0 and beta = 1")
print(ppp_joint_test)

UIP joint test: alpha = 0 and beta = 1
<F test: F=1.5037425481901094, p=0.22616706840712977, df_denom=129, df_num=2>

PPP joint test: alpha = 0 and beta = 1
<F test: F=0.09689839768811037, p=0.9077142203336739, df_denom=129, df_num=2>


In [14]:
beta_ci = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "beta_hat": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "beta_ci_low": [
        uip_model.conf_int().loc["uip_implied_change", 0],
        ppp_model.conf_int().loc["ppp_implied_change", 0]
    ],
    "beta_ci_high": [
        uip_model.conf_int().loc["uip_implied_change", 1],
        ppp_model.conf_int().loc["ppp_implied_change", 1]
    ]
})

beta_ci

,model,beta_hat,beta_ci_low,beta_ci_high
0,UIP,-2.119239,-6.180925,1.942447
1,PPP,0.693403,-1.305641,2.692448


In [15]:
uip_results.to_csv(OUT_DIR / "uip_regression_results_corrected.csv", index=False)
ppp_results.to_csv(OUT_DIR / "ppp_regression_results_corrected.csv", index=False)
summary.to_csv(OUT_DIR / "part2_regression_summary_corrected.csv", index=False)
theory_table.to_csv(OUT_DIR / "part2_theory_table_corrected.csv", index=False)
beta_ci.to_csv(OUT_DIR / "part2_beta_confidence_intervals_corrected.csv", index=False)

print("Saved corrected Part II outputs.")

Saved corrected Part II outputs.


In [18]:
# Final compact Part II regression table

uip_beta_eq_1_p = float(uip_model.t_test("uip_implied_change = 1").pvalue)
ppp_beta_eq_1_p = float(ppp_model.t_test("ppp_implied_change = 1").pvalue)

uip_joint_p = float(uip_model.f_test("const = 0, uip_implied_change = 1").pvalue)
ppp_joint_p = float(ppp_model.f_test("const = 0, ppp_implied_change = 1").pvalue)

part2_final_table = pd.DataFrame({
    "Model": ["UIP", "PPP"],
    "Equation": [
        "Δs_t = α + β(i_US,t - i_AUS,t) + ε_t",
        "Δs_t = α + β(π_US,t - π_AUS,t) + ε_t"
    ],
    "α": [
        uip_model.params["const"],
        ppp_model.params["const"]
    ],
    "p(α)": [
        uip_model.pvalues["const"],
        ppp_model.pvalues["const"]
    ],
    "β": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "p(β=0)": [
        uip_model.pvalues["uip_implied_change"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "95% CI β": [
        f"[{uip_model.conf_int().loc['uip_implied_change', 0]:.2f}, {uip_model.conf_int().loc['uip_implied_change', 1]:.2f}]",
        f"[{ppp_model.conf_int().loc['ppp_implied_change', 0]:.2f}, {ppp_model.conf_int().loc['ppp_implied_change', 1]:.2f}]"
    ],
    "p(β=1)": [
        uip_beta_eq_1_p,
        ppp_beta_eq_1_p
    ],
    "Joint p": [
        uip_joint_p,
        ppp_joint_p
    ],
    "R²": [
        uip_model.rsquared,
        ppp_model.rsquared
    ],
    "N": [
        int(uip_model.nobs),
        int(ppp_model.nobs)
    ]
})

# Round numeric columns for clean report presentation
numeric_cols = ["α", "p(α)", "β", "p(β=0)", "p(β=1)", "Joint p", "R²"]
part2_final_table[numeric_cols] = part2_final_table[numeric_cols].round(4)

part2_final_table

,Model,Equation,α,p(α),β,p(β=0),95% CI β,p(β=1),Joint p,R²,N
0,UIP,"Δs_t = α + β(i_US,t - i_AUS,t) + ε_t",-0.0031,0.5153,-2.1192,0.3039,"[-6.18, 1.94]",0.1311,0.2262,0.0082,131
1,PPP,"Δs_t = α + β(π_US,t - π_AUS,t) + ε_t",0.0009,0.7705,0.6934,0.4938,"[-1.31, 2.69]",0.7620,0.9077,0.0036,131


In [19]:
part2_final_table.to_csv(
    OUT_DIR / "part2_final_regression_table_corrected.csv",
    index=False
)